# CamScanner model training (auto)
Runs seg + enhance + cls training and TFLite conversion. Synthetic data fallbacks so no dataset downloads needed.

In [ ]:
import os, subprocess, sys, shutil
print('GPU:', os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip())
os.makedirs('/kaggle/working/camscanner', exist_ok=True)

In [ ]:
os.chdir('/kaggle/working')
if not os.path.isdir('camscanner/.git'):
    !git clone -q --depth 1 https://github.com/shubhambelbase/camscanner-train.git camscanner
os.chdir('camscanner')
print('repo ready')

In [ ]:
!pip install -q opencv-python-headless Pillow tqdm
import tensorflow as tf
print('tf', tf.__version__, 'gpus', tf.config.list_physical_devices('GPU'))

In [ ]:
os.makedirs('/out', exist_ok=True)
os.makedirs('/data', exist_ok=True)

In [ ]:
print('=== [1] seg train (synthetic fallback) ===')
!python train/seg_train.py --data /data --out /out --epochs 6
print('=== [2] enhance train ===')
!python train/enhance_train.py --out /out
print('=== [3] cls train (synthetic fallback) ===')
!python train/cls_train.py --data /data --out /out --epochs 6
print('=== [4] tflite ===')
!python train/convert_tflite.py --data /data --out /out
print('=== ALL_TRAINING_DONE ===')

In [ ]:
!ls -la /out/
!cp /out/*.tflite /out/cls_labels.txt /kaggle/working/ 2>/dev/null || cp /out/*.tflite /kaggle/working/
print('outputs copied to /kaggle/working')